# Get a Corpus Overview

Understand what's in your videos and images -- themes, subjects, patterns, and key statistics.
Use this when starting work with a new collection, feeding context to a downstream agent, or generating collection summaries for a UI.

In [ ]:
import json
import os

from twelvelabs import TwelveLabs, TextParam
from twelvelabs.types.text_param_format import TextParamFormat_JsonSchema

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
STORE_ID = os.environ.get("TWELVELABS_STORE_ID", "your_store_id")  # Replace with your knowledge store ID

client = TwelveLabs(api_key=API_KEY)

## Helper Functions

A utility to extract text content from a Jockey API response.

In [ ]:
def parse_response(response) -> str:
    """Extract text content from a Jockey response.

    Args:
        response: The ResponseObject returned by client.responses.create().

    Returns:
        The text content from the first message output, or an empty string
        if no message content is found.
    """
    for output in response.output:
        if output.type == "message":
            for content in output.content:
                return content.text
    return ""

## Plain Text Overview

The simplest approach: ask Jockey for a free-form summary of your videos and images.
This returns a natural-language overview covering main themes, recurring subjects, content types, and notable patterns.

In [ ]:
response = client.responses.create(
    knowledge_store_id=STORE_ID,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": (
                "Give me a comprehensive overview of these videos and images. "
                "Include main themes, recurring subjects, content types, "
                "and any notable patterns."
            ),
        }
    ],
)

print(parse_response(response))

## Structured Overview Schema

For programmatic consumption, define a JSON schema that tells Jockey exactly what fields to return.
This schema captures the total item count, dominant themes, content types, key subjects, notable patterns, and a narrative summary.

In [ ]:
OVERVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "total_items": {"type": "integer"},
        "themes": {"type": "array", "items": {"type": "string"}},
        "content_types": {"type": "array", "items": {"type": "string"}},
        "key_subjects": {"type": "array", "items": {"type": "string"}},
        "patterns": {"type": "array", "items": {"type": "string"}},
        "summary": {"type": "string"},
    },
}

## Structured Overview Request

Pass the schema via the `text` parameter to receive a structured JSON response.
The response text is valid JSON that can be parsed and used directly in downstream pipelines.

In [ ]:
response = client.responses.create(
    knowledge_store_id=STORE_ID,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": "Give me a structured overview of these videos and images.",
        }
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="corpus_overview", schema_=OVERVIEW_SCHEMA)
    ),
)

overview = json.loads(parse_response(response))

print(f"Items: {overview['total_items']}")
print(f"Themes: {', '.join(overview['themes'])}")
print(f"Content Types: {', '.join(overview['content_types'])}")
print(f"Key Subjects: {', '.join(overview['key_subjects'])}")
print(f"Patterns: {', '.join(overview['patterns'])}")
print(f"\nSummary: {overview['summary']}")

## Next Steps

- **[Search Videos](search_videos.ipynb)** -- find specific moments in your collection
- **[Extract Entities](extract_entities.ipynb)** -- list all people, places, objects, and concepts
- **[Find Organization Axes](find_organization_axes.ipynb)** -- discover the best ways to categorize your videos
- **[Enrich Content](enrich_content.ipynb)** -- get richer, domain-specific metadata

See also:
- [Structured Output Guide](https://docs.twelvelabs.io/v1.3/agents/guides/create-a-response/structured-output) -- more on JSON schema responses